# 04 · 真实数据准备与开盘成交检查

从指定 alpha 导出行情、特征和执行限制，再检查覆盖率、候选池变化以及无训练交易记账。数据库只读，输出写入项目的 `results/rl_experimental/`。

In [ ]:
from pathlib import Path
from dataclasses import asdict
import hashlib
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

repo_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src/res/rl_experimental").is_dir()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# 项目导入必须放在设置 repo_root 之后，支持从 notebooks 子目录运行。
from src.res.rl_experimental.data import PanelData, date_split  # noqa: E402
from src.res.rl_experimental.real_data import RealDataConfig, prepare_real_data  # noqa: E402

print("Python:", sys.executable)
print("Project:", repo_root)
# 在 notebook 的内核选择器中选择项目 .venv/bin/python。


## 准备真实数据

在项目主环境中依次运行，无需独立 RL 环境。以下示例读取本机已落库的 `pred@gru_day_V1`，默认先用 2025 年 1–2 月进行短实验；它不是新训练的 alpha，也尚未确认历史预测是否样本外。

04、05 的默认配置和输出位置相同。首次运行会导出数据，随后复用通过配置、文件指纹和数据契约检查的快照。修改源数据后需要主动重建；复用快照不会自动同步数据库。缺失日期会报错，不会自动换用其他 alpha。调整日期前应确认本机有回看期及最后决策日的下一交易日行情。


In [ ]:
ALPHA_SPEC = "pred@gru_day_V1"  # 改用其他 alpha 时显式修改这里
START_DATE = 20250102
END_DATE = 20250228  # 决策日期；另需下一交易日行情及起点之前的特征回看数据
ALPHA_SAMPLE_STATUS = "unknown"  # 确认历史预测为样本外后才改为 "out-of-sample"
REBUILD_SNAPSHOT = False  # True 会重新读取数据库并覆盖本配置的导出快照

if not ALPHA_SPEC.strip():
    raise ValueError("请填写 alpha，例如 pred@gru_day_V1")
snapshot_name = f"{ALPHA_SPEC.replace('@', '_')}_{START_DATE}_{END_DATE}"
SNAPSHOT_DIR = repo_root / "results/rl_experimental/snapshots" / snapshot_name
data_config = RealDataConfig(
    alpha=ALPHA_SPEC,
    start=START_DATE,
    end=END_DATE,
    output=SNAPSHOT_DIR,
    alpha_direction=1,
    alpha_lag=0,
    max_alpha_staleness=0,  # 每日严格匹配；低频信号须显式放宽
    min_listing_days=63,
    alpha_sample_status=ALPHA_SAMPLE_STATUS,
)
data_config.validate()
display(asdict(data_config))


In [ ]:
# 此 dry-run 检查 alpha 覆盖、证券数量和预计数组大小，不写文件。
# 行情、涨跌停、风格和历史信息的完整检查在下一单元正式导出时完成。
audit = prepare_real_data(data_config, dry_run=True)["audit"]
display(audit)
print(f"Estimated panel arrays: {audit['estimated_bytes'] / 1024**2:.1f} MiB")
print("准备时 DataFrame、中间数组和训练副本还会额外占用内存。")


In [ ]:
required_files = ["panel.npz", "manifest.json", "quality_report.json"]
existing = [(SNAPSHOT_DIR / name).is_file() for name in required_files]
expected_config = {**asdict(data_config), "output": str(SNAPSHOT_DIR)}

if REBUILD_SNAPSHOT or not any(existing):
    prepared = prepare_real_data(data_config)
    print("Exported:", prepared["panel"])
elif not all(existing):
    raise ValueError("快照文件不完整；检查输出目录后设置 REBUILD_SNAPSHOT=True 重新导出")
else:
    cached_manifest = json.loads((SNAPSHOT_DIR / "manifest.json").read_text(encoding="utf-8"))
    if cached_manifest["config"] != expected_config:
        raise ValueError("快照配置不匹配；请使用新目录或设置 REBUILD_SNAPSHOT=True")
    print("Reusing frozen snapshot:", SNAPSHOT_DIR)

manifest = json.loads((SNAPSHOT_DIR / "manifest.json").read_text(encoding="utf-8"))
quality = json.loads((SNAPSHOT_DIR / "quality_report.json").read_text(encoding="utf-8"))
with (SNAPSHOT_DIR / "panel.npz").open("rb") as handle:
    actual_sha256 = hashlib.file_digest(handle, "sha256").hexdigest()
if actual_sha256 != manifest["panel_sha256"]:
    raise ValueError("panel.npz 指纹不匹配；请重新导出快照")
panel = PanelData.load_npz(SNAPSHOT_DIR / "panel.npz")
if panel.contract() != manifest["panel_contract"]:
    raise ValueError("panel 与清单中的数据契约不匹配")
display(quality)
display(manifest["default_time_splits"])
print("Research status:", manifest["research_status"])


## 覆盖率与候选池

候选池仅由当日投资资格及 alpha 排名确定。图中的次日成交限制用于事后诊断，不能作为当日决策输入。

In [ ]:
candidate_sets = []
for t in range(len(panel.decision_dates)):
    eligible = np.flatnonzero(panel.investable[t])
    ordered = eligible[np.argsort(-panel.alpha[t, eligible], kind="stable")[:100]]
    candidate_sets.append(set(ordered.tolist()))

coverage = pd.DataFrame({
    "date": pd.to_datetime(panel.decision_dates.astype(str)),
    "investable": panel.investable.sum(axis=1),
    "candidates": [len(x) for x in candidate_sets],
    "candidate_entries": [0] + [
        len(current - previous)
        for previous, current in zip(candidate_sets, candidate_sets[1:])
    ],
    "cannot_buy_next_open": (panel.investable & ~panel.can_buy).sum(axis=1),
    "cannot_sell_next_open": (panel.investable & ~panel.can_sell).sum(axis=1),
    "stale_valuations": panel.valuation_stale.sum(axis=1),
})
coverage.plot(x="date", subplots=True, figsize=(12, 12))
plt.tight_layout()
display(coverage.head())


In [ ]:
# 仅统计当日有效股票，避免空槽位影响分布；*_missing 列反映缺失比例。
# 此处只是原始数据检查；PPO 的归一化在 train_experiment 中仅拟合训练段。
feature_summary = pd.DataFrame(
    panel.features[panel.investable], columns=panel.feature_names
).describe().T
display(feature_summary[["count", "mean", "std", "min", "max"]])


In [ ]:
event_rows = np.flatnonzero(
    (panel.investable & (~panel.can_buy | ~panel.can_sell)).any(axis=1)
)
t = int(event_rows[0]) if len(event_rows) else 0
candidate = np.flatnonzero(panel.investable[t])
candidate = candidate[np.argsort(-panel.alpha[t, candidate], kind="stable")[:100]]
print("Decision:", panel.decision_dates[t], "Execution:", panel.execution_dates[t])
display(pd.DataFrame({
    "secid": panel.stock_ids[candidate],
    "alpha_rank": panel.alpha[t, candidate],
    "can_buy_next_open": panel.can_buy[t, candidate],
    "can_sell_next_open": panel.can_sell[t, candidate],
    "overnight_return": panel.overnight_returns[t, candidate],
    "intraday_return": panel.intraday_returns[t, candidate],
}).sort_values(["can_buy_next_open", "can_sell_next_open"]))


## 无训练回放

先在训练段从现金开始回放两条基线，观察真实成交、费用和冻结持仓。`overnight_return` 是旧持仓从前收盘到开盘的组合收益；成交后再经历日内价格变化。04 不使用测试段挑策略，05 将统一比较测试段结果。

In [ ]:
from src.res.rl_experimental.baselines import replay_baseline, history_metrics
from src.res.rl_experimental.portfolio import PortfolioConstraints

constraints = PortfolioConstraints(candidate_count=100, max_holdings=50, max_weight=0.03, fee_rate=0.00035)
train_interval = date_split(panel)["train"]
baseline_histories = {
    kind: pd.DataFrame(replay_baseline(panel, train_interval, constraints, kind))
    for kind in ["alpha_equal_weight", "robust_top50"]
}
display(pd.DataFrame({
    kind: history_metrics(frame.to_dict("records"))
    for kind, frame in baseline_histories.items()
}).T)
ledger = baseline_histories["alpha_equal_weight"]
display(ledger[[c for c in [
    "decision_date", "return_end_date", "nav", "period_return", "overnight_return",
    "fee_estimate", "turnover", "cash_weight", "holdings", "frozen_holdings",
    "rejected_buys", "rejected_sells"
] if c in ledger]].head(10))
